### Local Settings ---- for installation on local computer ONLY

#### 1. Uv installation (local only, no need to redo if already done)


        https://docs.astral.sh/uv/getting-started/installation/


        `curl -LsSf https://astral.sh/uv/install.sh | sh`

        Python version 3.12 installation (highly recommended)
        `uv python install 3.12`


#### 3. Python env creation (local only)

        ```
        mkdir Projet_bias_mitigation
        cd Projet_bias_mitigation
        uv python pin 3.12
        uv init
        uv venv
        uv add numpy\
                torchvision\
                torch\
                torchmetrics\
                tensorboard\
                pillow\
                lightning\
                matplotlib\
                scikit_learn\
                ipykernel
        uv add pandas==2.2.2
        uv add setuptools==81.0
        ```

#### 4. In your folder "Projet_bias_mitigation"

        a. Copy your dataset and unzip it
                ```unzip YOUR_NAME.zip -d ./DATA/```
        b. Copy the "train_classifieur.py" in your folder

In [1]:
# 1. On connecte le Google Drive à la machine virtuelle Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. LE SECRET : On se déplace dans TON dossier (remplace par le bon nom)
# Attention, MyDrive correspond à la racine de ton Google Drive
%cd /content/drive/MyDrive/Projet_fairness

# 3. Vérification : On affiche les fichiers pour être sûr qu'on est au bon endroit
!ls

Mounted at /content/drive
/content/drive/MyDrive/Projet_fairness
DATA	      projet.ipynb	       train_classifieur_local_Carlos.ipynb
main.py       Projet_presentation.pdf  train_classifieur_local.ipynb
notation.png  README.md		       train_classifieur.py


In [3]:
!pip install torchmetrics pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 69.1 MB/s eta 0:00:00


- - -
# Projet Final

> Arthur LE GAL, Leonor MARTIN-NOURRY, Carlos CLEMENT

> LDDIM3

> Faculte des sciences d'Orsay

# Plan du projet:

> Introduction

> Préparation et analyse des données

> Application des méthodes de pre processing et étude de métriques de fairness

> Application des méthodes de post processing et étude de métriques de fairness

> Conclusion


![Notation Projet](./notation.png)

In [13]:
from train_classifieur import train_classifier, pred_classifier
from datetime import datetime

In [14]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [15]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Tesla T4
Memory Usage:
Allocated: 0.0 GB
Cached:    0.0 GB


In [16]:
%load_ext tensorboard
%tensorboard --logdir=$logdir

Launching TensorBoard...

KeyboardInterrupt: 

In [ ]:
%%time
# This cell show how to perform predictions with a train model
# the ckpt_path outputed in the previous cell refer to the best model obtained
# during the training, you can replace this by any .ckpt file obtained
pred_classifier(
    datadir=f"./DATA/",
    csv_in=f"./DATA/metadata.csv",
    csv_out=f"{logdir}/preds.csv",
    ckpt_path = ckpt_path
)

- - -
# Introduction

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


dataset = pd.read_csv("DATA/metadata.csv")
print(dataset.shape)
dataset.head()

(5581, 15)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,train_valid,label,WEIGHTS
0,00000008_000.png,Cardiomegaly,0,8,69,F,PA,2048,2500,0.171,0.171,NaN,train,malade,1
1,00000008_001.png,No Finding,1,8,70,F,PA,2048,2500,0.171,0.171,NaN,train,sain,1
2,00000008_002.png,Nodule,2,8,73,F,PA,2048,2500,0.168,0.168,NaN,train,malade,1
3,00000039_000.png,No Finding,0,39,76,M,PA,2500,2048,0.168,0.168,NaN,train,sain,1
4,00000039_001.png,No Finding,1,39,75,M,PA,2992,2991,0.143,0.143,NaN,train,sain,1


Comme au mi-projet, nous commençons par une prise en main du dataset.

Celui-ci contient 5581 lignes et 15 colonnes. On y retrouve les informations habituelles :

- l'identifiant de l'image radiographique,
- les diagnostics associés (ex : Mass|Nodule),
- le numéro de suivi du patient (0 = premier examen),
- l'identifiant du patient, son âge, son genre,
- la position de prise de vue (AP = antéro-postérieure, PA = postéro-antérieure),
- ainsi que les dimensions de l'image et la taille physique d'un pixel.

À cela s'ajoutent trois colonnes introduites pour le projet :

- train_valid (partition d'entraînement/validation),
- label (malade/sain)
- et WEIGHTS (poids d'échantillonnage).

Ce dataset nous permettra de chercher des corrélations,
des résultats notables et des biais grace a des informations médicales sur divers patients.

Pour simplifier l'analyse, nous ne retenons que 6 caractéristiques, en écartant l'identifiant image, les dimensions en pixels et la taille du pixel, qui n'apportent rien de pertinent ici.

- - -
# Preparation et analyse des données

> ## PREPARATION

In [19]:
dataset.dtypes #Pour voir avec quoi on travaille

,0
Image Index,object
Finding Labels,object
Follow-up #,int64
Patient ID,int64
Patient Age,int64
Patient Gender,object
View Position,object
OriginalImage[Width,int64
Height],int64
OriginalImagePixelSpacing[x,float64


On enleve Unnamed: 11, petite colonne d'erreur

In [20]:
dataset = dataset.drop(columns=["Unnamed: 11"])
print(dataset.columns.tolist())

['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'train_valid', 'label', 'WEIGHTS']


Au niveau de l'encodage des colonnes on procede de la meme facon que pour le mi-projet en ajoutant uniquement les colonnes supplementaires:

- train_valid : en one-hot encoding
- label : en one-hot encoding egalement
- WEIGHTS : pas de modification ici

In [21]:
selection = [
    "Finding Labels",
    "Follow-up #",
    "Patient ID",
    "Patient Age",
    "Patient Gender",
    "View Position",
    "train_valid",
    "label",
    "WEIGHTS"]

df = dataset[selection].copy()

df_test = df.copy() # Pour garder une copie avant modifications

for column in df.columns: #On verifie qu'il n'y ai pas de valeur NaN
    print(column, df[column].isnull().values.any())

labels = df["Finding Labels"].str.get_dummies(sep="|")
df = df.drop(columns=["Finding Labels"]).join(labels)

gender = pd.get_dummies(df["Patient Gender"], prefix="gender")
df = df.drop(columns=["Patient Gender"]).join(gender)

view = pd.get_dummies(df["View Position"], prefix="view")
df = df.drop(columns=["View Position"]).join(view)

train_valid = pd.get_dummies(df["train_valid"], prefix="train_valid")
df = df.drop(columns=["train_valid"]).join(train_valid)

label = pd.get_dummies(df["label"], prefix="label")
df = df.drop(columns=["label"]).join(label)

df.head()

Finding Labels False
Follow-up # False
Patient ID False
Patient Age False
Patient Gender False
View Position False
train_valid False
label False
WEIGHTS False


,Follow-up #,Patient ID,Patient Age,WEIGHTS,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,...,Pneumonia,Pneumothorax,gender_F,gender_M,view_AP,view_PA,train_valid_train,train_valid_valid,label_malade,label_sain
0,0,8,69,1,0,1,0,0,0,0,...,0,0,True,False,False,True,True,False,True,False
1,1,8,70,1,0,0,0,0,0,0,...,0,0,True,False,False,True,True,False,False,True
2,2,8,73,1,0,0,0,0,0,0,...,0,0,True,False,False,True,True,False,True,False
3,0,39,76,1,0,0,0,0,0,0,...,0,0,False,True,False,True,True,False,False,True
4,1,39,75,1,0,0,0,0,0,0,...,0,0,False,True,False,True,True,False,False,True


De la meme facon que lors du mi-projet on gere les ages car c'est la seule colonne avec des valeurs abherrantes.

In [22]:
print(f"Lignes avant : {len(df)}")
df = df[df["Patient Age"] <= 100]
print(f"Lignes après : {len(df)}")

Lignes avant : 5581
Lignes après : 5580


> ## ANALYSE

- - -
# Application des méthodes de pre processing et étude de métriques de fairness

In [23]:
# sauvegrade du dataset propre

chemin_csv_propre = "./DATA/metadata_cleaned.csv"
df.to_csv(chemin_csv_propre, index=False)

print(f"Le dataset nettoyé sauvegardé physiquement sous : {chemin_csv_propre}")

Le dataset nettoyé sauvegardé physiquement sous : ./DATA/metadata_cleaned.csv


In [ ]:
%%time
from datetime import datetime

nom_dossier = f"baseline_propre_{datetime.now().strftime('%Y_%m_%d_%H_%M_%S')}"
logdir = f"./expe_log/{nom_dossier}"

# 2. Lancement de l'entraînement sur le fichier de propre
ckpt_path, ckpt_score = train_classifier(
    logdir=logdir,
    datadir="./DATA/",
    csv="./DATA/metadata_cleaned.csv",
    weights_col=None,
    max_epochs=24
)

print(f"Entraînement terminé")
print(f"Meilleur modèle sauvegardé ici : {ckpt_path}")
print(f"Score (Balanced Accuracy) de validation : {ckpt_score}")

<mark> ATTENTION</mark> ne pas lancer la cellule suivante qui lance le modèle de prédiction sur un le dataset non propre.

In [7]:
%%time
# This cell show how to train a model
#
# You are going to train several time the models, with different weights
#
# The experiment folder name (logdir) suggested contains the timestamp
# you can change this to add some description in the name
# or add a file in the logdir folder to describe your intention and parameters
# Warning: For Colab users, the logdir needs to be on your drive (as in this example)
# It will allow you to keep your trained models, even if you get disconnected
#
# By default, to help you reproduce you experiments the csv file
# used for the training is copied in the logdir folder.

# This can take around 20-30min to run on PUIO computer
# And around 10 min on Colab with a GPU

dt = datetime.now()
timeStamp = dt.strftime('%Y_%m_%d_%H_%M_%S')
logdir = f"./expe_log/{timeStamp}"
ckpt_path, ckpt_score = train_classifier(
    logdir=logdir,
    datadir=f"./DATA/",
    csv=f"./DATA/metadata.csv",
    weights_col="WEIGHTS")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


./DATA/metadata.csv ./expe_log/2026_03_27_14_22_41/csv_in_WEIGHTS.csv
num_workers set to : 2
num_workers set to : 2
batch_size set to : 512
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 138MB/s]


Start training


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/Projet_fairness/expe_log/2026_03_27_14_22_41 exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

End of training 1710.2306010723114
CPU times: user 4min 31s, sys: 27.6 s, total: 4min 58s
Wall time: 28min 30s


Après avoir exécuter la cellule ci-dessus on a pu observer que l'entrainement du modèle s'est arrêté après 17 epochs et un temps de 28min 30s.

In [8]:
import pandas as pd

# 1. Remplace par le bon chemin vers ton fichier de prédictions
chemin_preds = "expe_log/2026_03_27_14_22_41/csv_in_WEIGHTS.csv"
df_preds = pd.read_csv(chemin_preds)

# Affichons les colonnes pour vérifier le nom exact de la colonne prédite (souvent 'pred', 'prediction' ou 'score')
print("Colonnes disponibles :", df_preds.columns.tolist())

Colonnes disponibles : ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'train_valid', 'label', 'WEIGHTS']


In [11]:
# --- VARIABLES À ADAPTER SELON TON CSV ---
colonne_prediction = 'label' # <- Change ça par le vrai nom de la colonne de prédiction de ton fichier
attribut_protege = 'View Position'     # 1 si PA, 0 si AP
# ----------------------------------------

def audit_modele(df, attr_protege, col_pred, priv_val='PA', unpriv_val='AP'):

    # On isole les données des deux groupes
    groupe_priv = df[df[attr_protege] == priv_val]
    groupe_unpriv = df[df[attr_protege] == unpriv_val]

    # L'ASTUCE ICI : On vérifie si le label est égal à 'malade' (en minuscules), puis on en fait la moyenne
    prob_pred_priv = (groupe_priv[col_pred] == 'malade').mean()
    prob_pred_unpriv = (groupe_unpriv[col_pred] == 'malade').mean()

    spd = prob_pred_unpriv - prob_pred_priv
    dir_ratio = prob_pred_unpriv / prob_pred_priv if prob_pred_priv > 0 else 0

    print(f"=== Audit du Modèle Baseline (Sans Pondération) ===")
    print(f"Le modèle prédit 'Malade' pour {prob_pred_priv*100:.1f}% des radios {priv_val}.")
    print(f"Le modèle prédit 'Malade' pour {prob_pred_unpriv*100:.1f}% des radios {unpriv_val}.")
    print("-" * 40)
    print(f"-> Statistical Parity Difference (SPD) : {spd:.4f}  (Objectif équité = 0.00)")
    print(f"-> Disparate Impact Ratio (DIR)        : {dir_ratio:.4f}  (Objectif équité = 1.00)")

# Lancement de l'audit
audit_modele(df_preds, attribut_protege, colonne_prediction)

=== Audit du Modèle Baseline (Sans Pondération) ===
Le modèle prédit 'Malade' pour 42.9% des radios PA.
Le modèle prédit 'Malade' pour 54.0% des radios AP.
----------------------------------------
-> Statistical Parity Difference (SPD) : 0.1113  (Objectif équité = 0.00)
-> Disparate Impact Ratio (DIR)        : 1.2594  (Objectif équité = 1.00)


L'audit du modèle sans pré-processing confirme que le réseau de neurones a appris et reproduit le biais d'acquisition présent dans les données d'entraînement. Le modèle a 11.13% (SPD = 0.1113) de chances en plus de prédire qu'un patient est malade si la radio est prise en position AP plutôt qu'en PA. Le Disparate Impact Ratio atteint 1.2594, ce qui dépasse le seuil critique d'équité (fixé mathématiquement entre 0.8 et 1.25). Le modèle Baseline est donc formellement considéré comme injuste vis-à-vis de la position de la radiographie. Pour rappel ce modèle à été entraîné sur le dataset original sans aucune modification.

- - -
# Application des méthodes de post processing et étude de métriques de fairness

- - -
# Analyse, compréhension de l'étude

- - -
# Conclusion